# Homework #2 (Due 02/03/2026, 11:59pm)
**ELEC70122: ML for Safety Critical Decision-Making**\
**Instructor: Sonali Parbhoo**\
**Spring 2026**

**Name:**

##Instructions:
**Submission Format:** Use this notebook as a template to complete your homework. Please intersperse text blocks (using Markdown cells) amongst python code and results -- format your submission for maximum readability. Your assignments will be graded for correctness as well as clarity of exposition and presentation -- a “right” answer by itself without an explanation or is presented with a difficult to follow format will receive no credit.

**Code Check:** Before submitting, you must do a "Restart and Run All" under "Kernel" in the Jupyter or colab menu. Portions of your submission that contains syntactic or run-time errors will not be graded.

**Libraries and packages:** Unless a problems specifically asks you to implement from scratch, you are welcomed to use any python library package in the standard Anaconda distribution.



**Total Marks: 40**

This tutorial is intentionally **theory and code integrated**.

You will:

- Derive reliability guarantees
- Empirically break them under shift
- Repair them via reweighting
- Audit sequential policies via OPE
- Enforce CMDP safety constraints

Work must include **code, plots, and written reasoning**.


# Global Learning Objectives

By the end you should be able to:

1. Diagnose reliability failure under covariate shift
2. Explain why uncertainty ≠ safety
3. Implement conformal calibration + weighted correction
4. Train unsafe RL policies and measure violations
5. Implement Doubly Robust OPE



# Mark Breakdown

| Section | Marks |
|--------|-------|
| Q1 Reliability Foundations | 8 |
| Q2 Conformal Under Shift | 12 |
| Q3 CMDPs + Unsafe RL | 9 |
| Q4 Doubly Robust OPE | 11 |
| **Total** | **40** |

---



In [4]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
np.random.seed(0)



# Q1 — Accuracy ≠ Safety (8 marks)

Modern decision systems rely on predictive uncertainty.

However:

> Accurate predictions can still produce unsafe decisions.


## Q1.1 Prediction vs Decision Risk (4 marks)

Formally define the differences between the following types of risks:

- Predictive risk
- Decision risk
- Constraint violation risk

Provide one healthcare or autonomous systems example that makes this distinction clear. Explain when one type of risk does not necessarily imply the others and when these risks may be entangled.

**Answer box:**





## Q1.2 Learning to Defer (4 marks)


Consider a probabilistic classifier \( f(x) \) that is allowed to **abstain (reject)** and defer its decision to an external expert (e.g., a human decision-maker or a downstream system). Let the cost of abstention be $ \lambda > 0$.

Answer the following:

1. Write the optimal decision rule that minimizes expected risk when the model can either: Predict a class label, or  Abstain and incur cost $\lambda$. Express the rule in terms of posterior class probabilities (or predictive confidence).
2. Explain how abstention reduces tail risk and improve safety and provide a real example
3. Give one distribution-shift failure mode example

**Answer box:**





# Q2 — Conformal Prediction Under Distribution Shift (12 marks)

In the previous exercise, you implemented split conformal prediction and evaluated empirical coverage and interval width.

However, your implementation assumed that the training, calibration, and test data are **exchangeable** (i.e., drawn from the same distribution). We will now study what happens under **covariate shift**, where:

$p_{\text{train}}(x) \neq p_{\text{test}}(x), \quad
p(y \mid x) \text{ unchanged.}$


This breaks the finite‑sample coverage guarantee because the empirical residual quantile computed on the calibration set is no longer representative of the test distribution. We will correct for this using **re-weighting**.


### (a) Create covariate shift (2 mark)
Modify the splits so that the calibration set and test set have different covariate distributions.
One simple way: split by a feature threshold, e.g. use higher values of one feature for test.

Report the empirical coverage and average width for $\alpha=0.1$ using your *unweighted* split conformal intervals.

Use the code below as a starting point:

In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.datasets import make_classification, make_regression

X, y = make_classification(
    n_samples=8000,
    n_features=20,
    n_informative=10,
    n_redundant=2,
    flip_y=0.02,
    class_sep=1.0,
    random_state=0,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
Xr, yr = make_regression(n_samples=4000, n_features=10, noise=15.0, random_state=0)


# -------------------------
# 1) Prepare splits (induce covariate shift)
# -------------------------
Xr_tr, Xr_temp, yr_tr, yr_temp = train_test_split(Xr, yr, test_size=0.4, random_state=0)

# Simple covariate shift example: pick a feature j and split temp by threshold
j = 0  # feature index to shift on (choose any)
thr = np.median(Xr_temp[:, j])

mask_test = Xr_temp[:, j] > thr # Optionally experiment with different thresholds to induce different shifts
Xr_te, yr_te = Xr_temp[mask_test], yr_temp[mask_test]
Xr_cal, yr_cal = Xr_temp[~mask_test], yr_temp[~mask_test]

# -------------------------
# 2) Fit base regressor
# -------------------------
base = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])
base.fit(Xr_tr, yr_tr)

pred_cal = base.predict(Xr_cal)
pred_te  = base.predict(Xr_te)

resid_cal = np.abs(yr_cal - pred_cal)

# -------------------------
# 3) Unweighted split conformal (from previous tutorial)
# -------------------------
def split_conformal_interval(pred, resid_cal, alpha=0.1):
    q = np.quantile(resid_cal, 1 - alpha)
    L = pred - q
    U = pred + q
    return L, U

Unweighted conformal prediction should fail under the induced distribution shift

### (b) Estimate density-ratio weights $w(x)=p_{\text{test}}(x)/p_{\text{cal}}(x)$ (4 marks)
Estimate weights using a **domain classifier**:

- Build a dataset with inputs $x$ from calibration (label 0) and test (label 1).
- Fit a probabilistic classifier $g(x)=P(D=1\mid x)$.
- Convert to importance weights:
$
w(x) = \frac{g(x)}{1-g(x)} \cdot \frac{1-\pi}{\pi},
$
where $\pi=P(D=1)$ is the class prior in your domain dataset.

Report min/mean/max of the weights on the calibration set.

Use the following code as a starting point.

In [7]:
# -------------------------
# 4) Estimate importance weights via domain classifier
# -------------------------
Xd = np.vstack([Xr_cal, Xr_te])
yd = np.hstack([np.zeros(len(Xr_cal)), np.ones(len(Xr_te))])

domain_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000))
])
domain_clf.fit(Xd, yd)

# g(x) = P(D=1 | x)
g_cal = domain_clf.predict_proba(Xr_cal)[:, 1]

# class prior in domain dataset
pi = np.mean(yd)  # = n_test / (n_cal + n_test)

# density ratio estimate on calibration points
eps = 1e-6
g_cal = np.clip(g_cal, eps, 1 - eps)
w_cal = (g_cal / (1 - g_cal)) * ((1 - pi) / pi)

# Optional: weight clipping to stabilize (you can decide if you allow this)
# w_cal = np.clip(w_cal, 0, np.quantile(w_cal, 0.99))

### (c) Implement weighted split conformal (4 marks)
Implement a function `weighted_split_conformal_interval(pred, resid_cal, w_cal, alpha)`
that uses the **weighted** $1-\alpha$-quantile of calibration residuals.

Recompute empirical coverage and average width for $\alpha=0.1$.
Compare to the unweighted results from part (a).

In [ ]:
# -------------------------
# 5) Weighted conformal TODOs
# -------------------------
def weighted_quantile(values, weights, q):
    """
    Return weighted quantile at level q in [0,1].
    TODO: implement.
    """
    raise NotImplementedError

def weighted_split_conformal_interval(pred, resid_cal, w_cal, alpha=0.1):
    """
    Use weighted (1-alpha)-quantile of resid_cal with weights w_cal.
    TODO: implement.
    """
    raise NotImplementedError

# -------------------------
# 6) Evaluate
# -------------------------
alpha = 0.1

L0, U0 = split_conformal_interval(pred_te, resid_cal, alpha=alpha)
cov0 = np.mean((yr_te >= L0) & (yr_te <= U0))
wid0 = np.mean(U0 - L0)

# Uncomment after implementing
# Lw, Uw = weighted_split_conformal_interval(pred_te, resid_cal, w_cal, alpha=alpha)
# covw = np.mean((yr_te >= Lw) & (yr_te <= Uw))
# widw = np.mean(Uw - Lw)

print("Unweighted coverage:", cov0, "width:", wid0)
# print("Weighted   coverage:", covw, "width:", widw)
print("Weights summary (cal): min/mean/max =", w_cal.min(), w_cal.mean(), w_cal.max())


### (d) Interpretation (2 marks)
In 3–5 sentences: when do you expect weighting to help, and what can go wrong
(e.g., extreme weights, domain classifier miscalibration, support mismatch)?


# Q3 — Constrained MDPs & Unsafe RL (9 marks)

So far, we have studied reliability at the prediction level.

We now move to **sequential decision-making**, where safety failures can accumulate over time. In many real-world systems e.g. healthcare treatment policies, robotics navigation, industrial control, decisions must trade off:

- Task reward (performance)
- Safety cost (risk, harm, constraint violations)

This setting is formalized as a **Constrained Markov Decision Process (CMDP)**:

$$
\max_\pi \; \mathbb{E}[R]
\quad \text{s.t.} \quad
\mathbb{E}[C] \le \alpha
$$

Where:

- R = cumulative reward
- C = cumulative safety cost
- α = allowable safety budget



## Environment description
Consider the following simple MDP setup with
- Actions $a \in \{0,1,2,3,\dots\}$.
- Some actions are **hazardous** (e.g., $ \{2,3\} $).
- Hazardous actions yield a higher expected reward but incur cost:
$
r \sim N(0,1) + 0.5 \cdot \mathbf{1}[a \in \text{hazards}],
\quad
c = \mathbf{1}[a \in \text{hazards}].
$

In [ ]:

class SimpleCMDP:
    def __init__(self):
        self.hazards={2,3}

    def step(self,a):
        r=np.random.normal()+ (0.5 if a in self.hazards else 0)
        c=1.0 if a in self.hazards else 0.0
        return r,c


### Q3.1 Why does reward maximization on its own violate safety? In plain English, explain why a standard reward-maximizing RL agent may learn unsafe behavior in this environment. (3 marks)

**Answer box:**

### Q3.2 Estimate Violation Probability (3 marks)

Using the `SimpleCMDP` environment, run multiple episodes of an **unconstrained agent** (i.e., one that selects reward-maximizing hazardous actions).

Your task is to estimate by completing the rollout code below the:

1. Mean episodic reward  
2. Mean episodic cost  
3. Probability of any violation  

Note that an episode is said to contain a violation if **any timestep** incurs cost \(c_t > 0\).

Thus, the violation probability is:

$
P(\text{any violation}) =
\frac{\text{# episodes with ≥ 1 violation}}{\text{total episodes}}.
$


In [1]:
import numpy as np

# ----- Environment (given) -----
class SimpleCMDP:
    def __init__(self):
        self.hazards = {2, 3}

    def step(self, a):
        r = np.random.normal() + (0.5 if a in self.hazards else 0.0)
        c = 1.0 if a in self.hazards else 0.0
        return r, c

# ----- Unconstrained policy (given) -----
def unconstrained_policy(env):
    """
    Reward-maximizing agent.
    In this environment, hazardous actions have higher mean reward.
    """
    return 2  # hazardous action


# ==============================
# TODO: ROLLOUT IMPLEMENTATION
# ==============================

def rollout_episode(env, policy_fn, horizon=50):
    """
    Run one episode.

    Returns:
        ep_reward : total reward
        ep_cost   : total cost
        violated  : True if any step had cost > 0
    """

    # TODO 1: initialize reward, cost, violation flag
    ep_reward = 0.0
    ep_cost = 0.0
    violated = False

    # TODO 2: simulate trajectory
    for t in range(horizon):

        # sample action
        a = policy_fn(env)

        # environment step
        r, c = env.step(a)

        # accumulate totals
        ep_reward += r
        ep_cost += c

        # check violation
        if c > 0:
            violated = True

    return ep_reward, ep_cost, violated


def estimate_violation_probability(env, policy_fn, n_episodes=200, horizon=50):
    """
    Run many episodes and compute:

      - mean episodic reward
      - mean episodic cost
      - probability of any violation
    """

    rewards = []
    costs = []
    violations = []

    for _ in range(n_episodes):
        R, C, V = rollout_episode(env, policy_fn, horizon)

        rewards.append(R)
        costs.append(C)
        violations.append(V)

    rewards = np.array(rewards)
    costs = np.array(costs)
    violations = np.array(violations)

    mean_reward = rewards.mean()
    mean_cost = costs.mean()
    prob_violation = violations.mean()

    return mean_reward, mean_cost, prob_violation


# ----- Run experiment -----
env = SimpleCMDP()

mean_R, mean_C, p_V = estimate_violation_probability(
    env,
    unconstrained_policy,
    n_episodes=500,
    horizon=50
)

print("Mean episodic reward:", mean_R)
print("Mean episodic cost:", mean_C)
print("Violation probability:", p_V)


Mean episodic reward: 25.119516996301275
Mean episodic cost: 50.0
Violation probability: 1.0


### Q3.3 Expected-Cost Lagrangian Constraint (3 marks)

In the previous question, you observed that maximizing reward alone leads to frequent safety violations. Based on the constrained MDP setup introduced at the start of Q3:

##### (a) Lagrangian objective (1 mark)

Write the **Lagrangian relaxation** of this constrained problem.

Your answer should introduce a multiplier $ \lambda \ge 0 $ and convert the constrained problem into an unconstrained objective.

##### (b) Behavioural effect (1 mark)

In plain English, explain how increasing  $\lambda$  changes the agent’s behaviour.

Specifically:

- What happens to hazardous actions?
- Why?

##### (c) Extremes (1 mark)

Describe the policy you would expect in the following limits:

1. $\lambda = 0$  
2. $ \lambda \to \infty$

Relate your answer to reward vs safety.

##### (d) Optional: Implement the Langrangian relaxation in the code above and observe what happens as $\lambda$ changes


**Answer box:**

## Q4 — Doubly Robust Off-Policy Evaluation (11 marks)

In many real-world decision problems, we do not collect data under the policy we wish to evaluate. Instead, logged data are generated by a behavior policy \( \pi_b \), while our goal is to estimate the expected reward of a different evaluation policy \( \pi_e \). As a result, expectations differ:

$
\mathbb{E}_{\pi_e}[R] \neq \mathbb{E}_{\pi_b}[R].
$

To correct this mismatch, we use importance sampling. Define the importance weight

$
\rho = \frac{\pi_e(a \mid s)}{\pi_b(a \mid s)}.
$

Using this weight, expectations under the evaluation policy can be written as

$
\mathbb{E}_{\pi_e}[R] = \mathbb{E}_{\pi_b}[\rho R].
$

Intuitively, we reweight observed outcomes so that logged data resemble samples drawn from $ \pi_e $.

However, pure importance sampling is unbiased but can suffer from high variance. Model-based approaches, which estimate reward using a learned model, have lower variance but may be biased if the model is misspecified. The **Doubly Robust (DR)** estimator combines both approaches by adding an importance-weighted correction to a direct reward model estimate. This yields an estimator that remains unbiased if either the model or the importance weights are correct, while typically achieving lower variance than importance sampling alone. This idea closely parallels weighted conformal prediction from earlier in the tutorial. There, we reweighted calibration data to match a shifted test distribution. Here, we reweight logged data from $ \pi_b $ to evaluate $ \pi_e $. In both settings, distribution mismatch is corrected via reweighting.



**(a)** Define the importance weight $ \rho $ and explain its role in off-policy evaluation.  
*(2 marks)*

**(b)** In plain English, describe why importance sampling has high variance.  
*(2 marks)*

**(c)** Explain why the Doubly Robust estimator is called “doubly robust.” What two sources of error can it tolerate?  
*(2 marks)*



**Answer box:**

**(d)** We will use the `SimpleCMDP` environment as a **single-state MDP** (i.e., a contextual bandit with no state). Logged data are collected under a **behavior policy** $ \pi_b $, but we want to estimate the expected reward of a different **evaluation policy** $ \pi_e $. **(5 marks)**

You will implement three off-policy estimators for $V(\pi_e)=\mathbb{E}_{a\sim \pi_e}[r(a)]$:

- Direct Method (DM): model-based estimate (potentially biased)
- Importance Sampling (IS): unbiased (high variance)
- Doubly Robust (DR): combines DM + IS correction

For this bandit setting, the DR estimator can be written as:

$
\widehat V_{\mathrm{DR}}
= \frac{1}{n}\sum_{i=1}^n \left[
\hat r(\pi_e) \;+\; \rho_i \big(r_i - \hat r(a_i)\big)
\right],
\quad
\rho_i=\frac{\pi_e(a_i)}{\pi_b(a_i)}.
$

Here $ \hat r(a) $ is a learned reward model (e.g., per-action mean reward), and
$\hat r(\pi_e)=\sum_a \pi_e(a)\hat r(a)$.


**(i)** Generate logged data under \( \pi_b \): actions \(a_i\) and rewards \(r_i\). *(1 mark)*  
**(ii)** Fit a simple reward model \( \hat r(a) \) from the logged data. *(1 mark)*  
**(iii)** Implement DM and IS estimators. *(1 mark)*  
**(iv)** Implement the Doubly Robust estimator using the formula above. *(2 marks)*


In [ ]:

### Starter code (complete the TODOs)

import numpy as np

# ----- Environment (given) -----
class SimpleCMDP:
    def __init__(self):
        self.hazards = {2, 3}

    def step(self, a):
        r = np.random.normal() + (0.5 if a in self.hazards else 0.0)
        c = 1.0 if a in self.hazards else 0.0
        return r, c


# ----- Policies (given) -----
def make_behavior_policy(n_actions):
    """
    Example behavior policy pi_b: mostly safe actions, sometimes hazards.
    Returns a probability vector of shape (n_actions,).
    """
    pi_b = np.ones(n_actions)
    pi_b = pi_b / pi_b.sum()
    return pi_b

def make_evaluation_policy(n_actions):
    """
    Example evaluation policy pi_e: more mass on hazardous actions (higher mean reward).
    Returns a probability vector of shape (n_actions,).
    """
    pi_e = np.ones(n_actions)
    pi_e = pi_e / pi_e.sum()
    return pi_e

def sample_action(pi, rng):
    return rng.choice(len(pi), p=pi)

def collect_bandit_data(env, pi_b, n=2000, seed=0):
    """
    Collect logged bandit data under behavior policy pi_b.

    Returns:
        actions: array shape (n,)
        rewards: array shape (n,)
        (ignore costs for this OPE question)
    """
    rng = np.random.default_rng(seed)

    actions = np.zeros(n, dtype=int)
    rewards = np.zeros(n, dtype=float)

    for i in range(n):
        # TODO (a): sample action a_i ~ pi_b
        # TODO (a): step env, record reward
        raise NotImplementedError

    return actions, rewards


def fit_reward_model_per_action(actions, rewards, n_actions):
    """
    Fit a simple reward model r_hat(a) = average reward observed for action a.
    Use a small fallback (e.g., global mean) for actions never taken.

    Returns:
        r_hat: array shape (n_actions,)
    """
    r_hat = np.zeros(n_actions, dtype=float)

    # TODO (b): compute per-action means from logged data
    # Hint: for each a, take rewards[actions==a].mean() if there are any.
    raise NotImplementedError

    return r_hat


def estimate_dm(pi_e, r_hat):
    """
    Direct Method: V_DM = sum_a pi_e(a) * r_hat(a)
    """
    # TODO (c): implement
    raise NotImplementedError


def estimate_is(actions, rewards, pi_b, pi_e):
    """
    Importance Sampling: V_IS = (1/n) * sum_i rho_i * r_i
    where rho_i = pi_e(a_i)/pi_b(a_i)
    """
    # TODO (c): compute rho for each logged action and return IS estimate
    raise NotImplementedError


def estimate_dr(actions, rewards, pi_b, pi_e, r_hat):
    """
    Doubly Robust:
      V_DR = (1/n) * sum_i [ r_hat(pi_e) + rho_i * (r_i - r_hat(a_i)) ]
    where r_hat(pi_e) = sum_a pi_e(a)*r_hat(a)
    """
    # TODO (d1): compute r_hat(pi_e)
    # TODO (d2): compute rho_i for each i
    # TODO (d3): compute DR estimate
    raise NotImplementedError


# ----- Run experiment (should work after TODOs) -----
env = SimpleCMDP()
n_actions = 5

pi_b = make_behavior_policy(n_actions)
pi_e = make_evaluation_policy(n_actions)

actions, rewards = collect_bandit_data(env, pi_b, n=5000, seed=0)
r_hat = fit_reward_model_per_action(actions, rewards, n_actions)

v_dm = estimate_dm(pi_e, r_hat)
v_is = estimate_is(actions, rewards, pi_b, pi_e)
v_dr = estimate_dr(actions, rewards, pi_b, pi_e, r_hat)

print("V_DM:", v_dm)
print("V_IS:", v_is)
print("V_DR:", v_dr)
